In [10]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [2]:
df_raw = spark.read.load('hdfs://localhost:9000/user/sbriones/ca1/eirgrid_raw/eirgrid_ALL_2014_2026.csv', format="csv", header="true", sep=',', inferSchema=True)
df_raw.printSchema()

root
 |-- EffectiveTime: timestamp (nullable = true)
 |-- SYSTEM_DEMAND: double (nullable = true)
 |-- GEN_EXP: double (nullable = true)
 |-- WIND_ACTUAL: double (nullable = true)
 |-- CO2_INTENSITY: double (nullable = true)
 |-- CO2_EMISSIONS: double (nullable = true)
 |-- solar_actual: double (nullable = true)
 |-- solar_forecast: double (nullable = true)
 |-- INTER_EWIC: double (nullable = true)
 |-- INTER_MOYLE: double (nullable = true)
 |-- INTER_NET: double (nullable = true)
 |-- INTER_GRNLK: double (nullable = true)



In [3]:
df_raw.show(5)

+-------------------+-------------+-------+-----------+-------------+-------------+------------+--------------+----------+-----------+---------+-----------+
|      EffectiveTime|SYSTEM_DEMAND|GEN_EXP|WIND_ACTUAL|CO2_INTENSITY|CO2_EMISSIONS|solar_actual|solar_forecast|INTER_EWIC|INTER_MOYLE|INTER_NET|INTER_GRNLK|
+-------------------+-------------+-------+-----------+-------------+-------------+------------+--------------+----------+-----------+---------+-----------+
|2014-01-01 00:00:00|       3758.0| 3390.0|     1386.0|        383.0|       1346.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:15:00|       3724.0| 3359.0|     1348.0|        383.0|       1335.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:30:00|       3666.0| 3301.0|     1272.0|        383.0|       1314.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:45:00|       3611.0| 3246.0|     1273.0|   

In [11]:
  # Inspect columns before cleaning                                                                                                                          
  # solar_actual and solar_forecast are empty for most of the dataset (pre-2020)                                                                             
  # GEN_EXP and CO2_EMISSIONS are not required for LSTM wind forecasting                                                                                                                                                                                                                 
                                                                                                                                                             
  # Select only relevant columns for LSTM wind power forecasting                                                                                             
  # Source: EirGrid Smart Grid Dashboard data dictionary                                                                                                     
COLS_TO_KEEP = ["EffectiveTime", "SYSTEM_DEMAND", "GEN_EXP", "WIND_ACTUAL", "CO2_INTENSITY"]                                                                          
                                                                                                                                                             
df_selected = df_raw.select(COLS_TO_KEEP)                                                                                                                                                                                                                 
df_selected.show(5) 

+-------------------+-------------+-------+-----------+-------------+
|      EffectiveTime|SYSTEM_DEMAND|GEN_EXP|WIND_ACTUAL|CO2_INTENSITY|
+-------------------+-------------+-------+-----------+-------------+
|2014-01-01 00:00:00|       3758.0| 3390.0|     1386.0|        383.0|
|2014-01-01 00:15:00|       3724.0| 3359.0|     1348.0|        383.0|
|2014-01-01 00:30:00|       3666.0| 3301.0|     1272.0|        383.0|
|2014-01-01 00:45:00|       3611.0| 3246.0|     1273.0|        391.0|
|2014-01-01 01:00:00|       3543.0| 3178.0|     1236.0|        393.0|
+-------------------+-------------+-------+-----------+-------------+
only showing top 5 rows
